# Day 24 — Advanced Context Management & FastAPI

**Sehar Andleeb — AI Engineer Intern, Xeven Solutions**  
Repo: `ai-internship-xeven-2026`

Two sessions today:

1. **Morning — Advanced Context Management:** conversational RAG with memory. Recent exchanges kept verbatim, older ones summarized (context compression), so a long chat stays inside the context window.
2. **Afternoon — FastAPI:** built a first API from scratch (health, path/query params, Pydantic body, auto-validation, Swagger docs), then wrapped the RAG system in a `POST /ask` endpoint with startup-once loading and proper HTTP error codes.

Environments: `.venv312` (Python 3.12) for anything using FAISS / RAG; `.venv` (Python 3.13) for the FastAPI-only basics. Embeddings use the offline `DeterministicEmbedder` (numpy only) to avoid the PyTorch DLL crash seen earlier in the internship.

### Setup — import path

The reusable modules live in `day24/scripts/`, one folder below this notebook. This cell adds that folder to the import path so the cells below can import them. Run this first.

**Note:** the memory-logic cell runs fully offline. The conversational RAG cell and the FastAPI sections are documented with their real outputs from 18 June 2026 — running the RAG cell would make a live Groq call, so re-run it only if you intend that.

In [1]:
import os
import sys

# Modules live in day24/scripts/; add that to the import path.
sys.path.insert(0, os.path.abspath("scripts"))
print("scripts/ added to path")

scripts/ added to path


## Morning Research — Advanced Context Management

Sources consulted **18 June 2026** across ChatGPT, Gemini, Claude, and two articles.

**Articles:**
1. T. Soni, *The Ultimate Guide to LLM Memory: From Context Windows to Advanced Agent Memory Systems*, Medium, 2025 — https://medium.com/@sonitanishk2003/the-ultimate-guide-to-llm-memory-from-context-windows-to-advanced-agent-memory-systems-3ec106d2a345
2. *Top 6 Techniques to Manage Context Length in LLMs*, Agenta — https://agenta.ai/blog/top-6-techniques-to-manage-context-length-in-llms

| Topic | ChatGPT | Gemini | Claude | Articles |
|---|---|---|---|---|
| Context window limits | Max tokens (input+output) the model processes at once; common sizes 4K/8K/128K (confirm) | Going over causes errors or silent truncation; budget shared by prompt, history, docs, answer (confirm) | A single shared token budget; everything (instructions, retrieved docs, chat history, question, and the reply) must fit together | Window measured in tokens; ~4K tokens ≈ 3000 words; info outside is forgotten for that pass |
| Dynamic context injection | Insert relevant content per request at inference instead of a fixed prompt (confirm) | In RAG, retrieve and slot in only the most relevant chunks for this question (confirm) | Chosen fresh per request, not baked in; the retriever decides what to inject right before the call | Retrieval/relevance selection loads only what is needed into the window on demand |
| Memory types | Buffer (verbatim), summary (compressed), entity (tracked facts) (confirm) | Buffer grows unbounded; summary saves tokens but loses detail; entity tracks key items (confirm) | Three tiers: buffer keeps exact text, summary compresses old turns, entity keeps a running fact sheet; pick by cost vs fidelity | Short-term = context window; long-term = external store; summary buffer is the common hybrid |
| Context compression | Summarize old messages, keep recent ones detailed (confirm) | Hybrid summary-buffer triggers on a token limit and folds oldest turns into a summary (confirm) | Keep last N exchanges verbatim, compress everything older into one running summary; bounds the window regardless of length | Recursive summarization of evicted turns (MemGPT-style); trades detail for bounded size |


### Note on implementation choice

The articles describe LangChain's `ConversationSummaryBufferMemory` as the standard class for the recent-verbatim + older-summarized pattern. That class is **deprecated in LangChain 0.3.x**, so I implemented the same strategy from scratch using the modern **message-list pattern**: memory is a plain list of `(question, answer)` pairs, the most recent `MAX_RECENT = 10` are kept verbatim, and older overflow is summarized by Groq and dropped.

### Task 1 — Conversation memory (pruning logic)

`ConversationMemory` stores recent pairs verbatim and routes overflow to a caller-supplied summarizer. The summarizer is passed in from outside, so the pruning logic can be tested with a dummy function before any LLM is involved.

In [2]:
from conversation_memory import ConversationMemory

# Window of 3 for a quick demo; real default is 10.
m = ConversationMemory(max_recent=3)
for i in range(5):
    m.add(f"q{i}", f"a{i}")

print("before prune, recent count:", len(m.recent))
print("overflow:", m.overflow())
m.prune(lambda s, pairs: s + f" [summarized {len(pairs)} old pairs]")
print("after prune, recent count:", len(m.recent))
print("summary:", repr(m.summary))
print("kept:", m.recent)

before prune, recent count: 5
overflow: [('q0', 'a0'), ('q1', 'a1')]
after prune, recent count: 3
summary: ' [summarized 2 old pairs]'
kept: [('q2', 'a2'), ('q3', 'a3'), ('q4', 'a4')]


The two oldest pairs (`q0`, `q1`) were routed to the summarizer; the newest three survived verbatim. Pruning plumbing confirmed with zero LLM cost.

### Task 1 — Conversational RAG with memory (live conversation)

Combines the offline retriever, the Groq LLM (`llama-3.3-70b-versatile`), and the memory. The key test is **Turn 2**: *"How do I prevent it?"* has no keyword, but memory carries the topic from Turn 1 so the model resolves *"it"* = overfitting.

*(Output below is the documented result of running `conversational_rag.py` on 18 June 2026.)*

In [3]:
from rag_core import (
    DeterministicEmbedder, SAMPLE_DOCS, build_vector_store,
)
from conversation_memory import ConversationMemory
from conversational_rag import build_llm, chat_turn

embedder = DeterministicEmbedder()
index, docs = build_vector_store(SAMPLE_DOCS, embedder)
llm = build_llm()
memory = ConversationMemory()

for turn, q in enumerate(
    ["What is overfitting?", "How do I prevent it?",
     "What is a transformer?"], start=1):
    answer, _ = chat_turn(q, index, docs, embedder, llm, memory)
    print(f"=== Turn {turn} ===")
    print("Q:", q)
    print("A:", answer, "\n")

=== Turn 1 ===
Q: What is overfitting?
A: Overfitting happens when a model memorizes the training data and performs poorly on new, unseen examples. 

=== Turn 2 ===
Q: How do I prevent it?
A: You can prevent overfitting by using regularization techniques like dropout and weight decay, which discourage overly complex models. 

=== Turn 3 ===
Q: What is a transformer?
A: A transformer is a neural architecture built on attention that processes all tokens in a sequence in parallel. 



Turn 2 answered about *overfitting* despite the bare pronoun *"it"* — proof the memory layer works. Honest caveat: the retriever scores follow-ups weakly (no shared keyword), so memory, not retrieval, rescues the answer. A production system would add query rewriting; that is beyond Day 24 scope.

## Afternoon Research — FastAPI

Sources consulted **18 June 2026** across ChatGPT, Gemini, Claude, and two tutorials.

**Tutorials:**
1. *Request Body*, official FastAPI docs (tiangolo) — https://fastapi.tiangolo.com/tutorial/body/
2. *FastAPI Basics: Path Parameters, Query Parameters and Request Body*, gpttutorpro — https://gpttutorpro.com/fastapi-basics-path-parameters-query-parameters-and-request-body/

| Topic | ChatGPT | Gemini | Claude | Tutorials |
|---|---|---|---|---|
| What is FastAPI | Modern async Python web framework with automatic docs (confirm) | Built on Starlette + Pydantic; high performance (confirm) | A framework for building APIs that reads your type hints to validate data and auto-generate interactive docs | Uses type hints + Pydantic + OpenAPI; validates and documents automatically |
| Key features | Type hints, Pydantic validation, Swagger/ReDoc auto-docs (confirm) | OpenAPI schema generation, editor support, async (confirm) | Type hints become validation for free; a wrong type returns a 422 before your code runs; `/docs` is generated from code + docstrings | Pydantic models declare/validate/document request bodies; errors return JSON under `detail` |
| REST concepts | GET read, POST create, PUT update, DELETE remove (confirm) | Path params identify a resource; query params filter; body carries JSON (confirm) | Path param = part of URL (`/items/5`); query param = after `?`; body = JSON validated by a Pydantic model | Singular type → query param; Pydantic model → request body; all three combinable in one route |
| Why FastAPI for AI | Async suits slow LLM calls; easy to deploy (confirm) | Handles concurrent requests while waiting on I/O (confirm) | Async lets the server serve others while one request waits on Groq; type validation guards model inputs cleanly | Async I/O enables high concurrency; integrates with ML model serving patterns |


### Task 2 — First FastAPI application (`day24/api/main.py`)

A health check, a path parameter with an optional query parameter, and a POST endpoint with Pydantic body validation. Run with `uvicorn main:app --reload` from `day24/api/` in `.venv`.

In [4]:
"""day24/api/main.py — shown for reference; runs via uvicorn."""
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Day 24 API", version="0.1.0")

ITEMS = {1: {"name": "Notebook", "price": 3.5},
         2: {"name": "Pen", "price": 1.0}}


class Item(BaseModel):
    name: str
    price: float


@app.get("/health")
def health():
    return {"status": "ok"}


@app.get("/items/{item_id}")
def get_item(item_id: int, q: str | None = None):
    item = ITEMS.get(item_id)
    if item is None:
        return {"error": "item not found", "item_id": item_id}
    result = {"item_id": item_id, "item": item}
    if q is not None:
        result["q"] = q
    return result


@app.post("/items")
def create_item(item: Item):
    new_id = max(ITEMS) + 1 if ITEMS else 1
    ITEMS[new_id] = {"name": item.name, "price": item.price}
    return {"item_id": new_id, "item": ITEMS[new_id]}

**Tested results (18 June 2026):**

| Request | Response | Code |
|---|---|---|
| `GET /items/1` | `{"item_id":1,"item":{"name":"Notebook","price":3.5}}` | 200 |
| `GET /items/99` | `{"error":"item not found","item_id":99}` | 200 |
| `GET /items/abc` | `{"detail":[{"type":"int_parsing",...}]}` | 422 |
| `POST /items` `{"name":"Eraser","price":0.75}` | `{"item_id":3,"item":{...}}` | 200 |

The `422` on `/items/abc` is automatic: the `item_id: int` hint made FastAPI reject the non-integer before the function ran — no validation code written by hand.

### Task 3 — RAG API endpoint (`day24/api/rag_api.py`)

Wraps the conversational RAG system in `POST /ask`. The embedder, FAISS index, and Groq LLM are built **once at startup** via a `lifespan` handler (the roadmap's `@app.on_event("startup")` is deprecated, so `lifespan` is used instead) and reused for every request. Returns `{answer, sources, confidence}` with 400 / 404 / 500 error handling. Run with `uvicorn rag_api:app --reload` from `day24/api/` in `.venv312`.

**Startup log (proves build-once):**

```
RAG system ready: 8 documents indexed.
INFO:     Application startup complete.
```

**Tested `/ask` results (18 June 2026):**

| Request body | Result | Code |
|---|---|---|
| `{"question": "What is overfitting?"}` | answer + 3 sources, `confidence` 0.289 | 200 |
| `{"question": "   "}` | `{"detail":"Question is empty."}` | 400 |
| `{"question": "How do I prevent it?"}` (separate call) | answers about preventing overfitting | 200 |

The third row is the headline result: *"it"* is undefined in that request, yet the answer is correct — because the `lifespan`-built `ConversationMemory` is shared across requests, so the server remembers the earlier `/ask` call. Conversational RAG with memory, working over HTTP, across independent requests.

Note: `confidence` is the top FAISS retrieval similarity score, **not** the LLM's certainty in its answer.

## Summary

- Built conversational RAG with a from-scratch memory layer (recent verbatim + older summarized), proving follow-up resolution and pruning separately.
- Learned FastAPI from first principles: routing, path/query params, Pydantic body validation, automatic 422s, and Swagger docs.
- Wrapped the RAG system in a production-shaped `POST /ask` endpoint with startup-once loading and HTTP error handling, and demonstrated memory persisting across separate API calls.
- Deviated from two deprecated roadmap APIs (`ConversationBufferMemory`, `@app.on_event`) in favour of their current replacements, documenting why.